# code message disclosure

> Exploring compacted view of code messages.

In [ ]:
#| export
#| hide
import re, json
from html import escape
from pathlib import Path
from typing import TypeAlias
from IPython import get_ipython
from mistletoe import Document
from mistletoe.block_token import CodeFence
from mistletoe.markdown_renderer import MarkdownRenderer
from nbdev.process import extract_directives
from fastcore.foundation import patch, adict, AttrDict as AD
from fastcore.meta import delegates
from dialoghelper.core import add_msg, read_msg, find_msgs, msg_idx, run_msg, toggle_header, ast_py, find_dname
from pote.dutil import in_dialog

In [ ]:
#| hide
from fastcore.test import *
from pote.dutil import link_msg

## Code message disclosure

You know all about SolveIt, right?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes! SolveitAI is the AI assistant built into the solveit dialog engineering app by fast.ai. It supports three message types — `code`, `note`, and `prompt` — in a persistent Python environment backed by ipykernel. Users can edit any message in the dialog at any time, which rewrites history from my perspective.

What are you working on?

<details class='token-usage-details'><summary>$0.0673</summary>

`total=11,825 | in=11,741 | out=84 | cached=0.0% | cache_new=11,135 | searches=0 | $0.0673`

</details>


I need a function to given the python source code of a `code` message returns tree kinds of outputs with increased levels of details (disclosure).
- First level return just the function signatures (and optionaly module level definitions)
- Second level returns signatures + docstrings and `fastcore`'s `docments`
- Third level returns everything in the first two levels plus the full source code of each function.

This "disclosure" levels will be used downstream to generate digest of SolveIt dialogs (or Jupyter notebooks) with progressive levels of detail mostly for givin LLM context.

Before coding, please ask me your doubts until we reach an agreement then I will give you de go ahead. You can also propose approaches and improvements.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great idea — a few questions before we start:

1. **Output format**: Should the function return a plain string (reconstructed Python source-like text), or a structured object (e.g. a dict/list of dicts per definition)? A string is simpler for LLM context; a structured format gives more flexibility downstream.

2. **Static vs. live**: Should this work purely on source text (AST parsing, no execution), or can we assume the code has already been executed in the kernel so we can use `inspect` / fastcore's `docments` on live objects? Static is safer but `docments` works on live functions.

3. **Module-level definitions**: What exactly should be included — just assignments (`x = 5`, `TypeAlias`, `__all__`), or also class definitions and decorators like `@patch`?

4. **Classes**: Should class bodies be handled? e.g. show class signature + method signatures at level 1, add docstrings at level 2, full bodies at level 3?

5. **`@patch`ed methods**: These are common in fast.ai-style code. Should they be treated like regular function defs?

6. **API shape**: One function with a `level: int` param, or a function returning all three levels at once (e.g. as a tuple or dict)? Returning all three at once might be more useful for building digests.

7. **Directives**: The code already imports `extract_directives` from nbdev — should `#| hide` / `#| export` etc. be respected (e.g. skip hidden cells)?

<details class='token-usage-details'><summary>$0.0872</summary>

`total=12,924 | in=12,041 | out=883 | cached=0.0% | cache_new=11,699 | searches=0 | $0.0872`

</details>


1. **Output format**: Should the function return a plain string (reconstructed Python source-like text).

2. **Static vs. live**: Should this work purely on source text (AST parsing, no execution). We'll explore the two versions, but first static using AST or any pther similar library (i.e., `tree-sitter`).

3. **Module-level definitions**: What exactly should be included — just assignments (`x = 5`, `TypeAlias`, `__all__`) Whatever is at module level (root): classes,definitions, functions, etc.

4. **Classes**: Classes methods should be handled with the level of detail requested.

5. **`@patch`ed methods**: These are common in fast.ai-style code. Should they be treated like regular function defs? Yes.

6. **API shape**: One function with a `level: int` param, or a function returning all three levels at once (e.g. as a tuple or dict)? Returning all three at once might be more useful for building digests. => Don't know yet, not important. As we'll develop modularly we can always write a higher level function afterwards we write and test basic functions.

7. **Directives**: The code already imports `extract_directives` from nbdev — should `#| hide` / `#| export` etc. be respected (e.g. skip hidden cells)? No, they will be stripped (are handled at another level)

---

What references will you need to code confidently?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good set of answers — clear enough to get started. Before diving in, I'd like to look up a few things to code confidently:

1. **`ast-grep-py`** — it's available in the instance and could be a cleaner alternative to Python's `ast` for pattern matching. I want to check if it's worth using here or if plain `ast` is better for this task.

2. **`ast.unparse`** — I'm fairly confident about this for reconstructing signatures from AST nodes, but I'd like to double-check its behaviour on `args`, decorators, and annotations in Python 3.12.

3. **fastcore `docments`** — I know it works on live objects, but I want to check if there's a static/source variant, since you mentioned it for level 2. If not, we'll parse annotation comments (`#: ...` style) from the AST ourselves.

Shall I go ahead and look those up now?

<details class='token-usage-details'><summary>$0.0909</summary>

`total=13,667 | in=12,799 | out=868 | cached=0.0% | cache_new=12,282 | searches=0 | $0.0909`

</details>


In [ ]:
ast_py??
ast_grep??
msg_ast_replace??


```python
def ast_py(code:str):
    "Get an SgRoot root node for python `code`"
    from ast_grep_py import SgRoot
    return SgRoot(code, "python").root()
```

**File:** `/usr/local/lib/python3.12/site-packages/dialoghelper/core.py`


```python
@llmtool
def ast_grep(
    pattern:str, # ast-grep pattern to search, e.g "post($A, data=$B, $$$)"
    path:str=".", # path to recursively search for files
    lang:str="python" # language to search/scan
): # json format from calling `ast-grep --json=compact
    """Use `ast-grep` to find code patterns by AST structure (not text).

    Pattern syntax:
    - $VAR captures single nodes, $$$ captures multiple
    - Match structure directly: `def $FUNC($$$)` finds any function; `class $CLASS` finds classes regardless of inheritance
    - DON'T include `:` - it's concrete syntax, not AST structure
    - Whitespace/formatting ignored - matches structural equivalence

    Examples: `import $MODULE` (find imports); `$OBJ.$METHOD($$$)` (find method calls); `await $EXPR` (find await expressions)

    Useful for: Refactoring—find all uses of deprecated APIs or changed signatures; Security review—locate SQL queries, file operations, eval calls; Code exploration—understand how libraries are used across codebase; Pattern analysis—find async functions, error handlers, decorators; Better than regex—handles multi-line code, nested structures, respects syntax"""
    cmd = f"ast-grep --pattern '{pattern}' --lang {lang} --json=compact"
    if path != ".": cmd = f"cd {path} && {cmd}"
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return json.loads(res.stdout) if res.stdout else res.stderr
```

**File:** `/usr/local/lib/python3.12/site-packages/dialoghelper/core.py`


```python
def _ast_replace(
    text:str,
    pattern:str, # AST pattern to match, e.g. 'print($A)'; use $VAR for single nodes, $$$ for multiple
    rewrite:str, # Replacement template with metavariables expanded, e.g. 'log($A)'
    lang:str='python' # Source language (see ast-grep --help for supported languages)
 ):
    """Replace code by AST pattern using ast-grep.

    Pattern syntax:
    - $VAR captures single nodes, $$$ captures multiple
    - Match structure directly: `def $FUNC($$$)` finds any function; `class $CLASS` finds classes regardless of inheritance
    - DON'T include `:` - it's concrete syntax, not AST structure
    - Whitespace/formatting ignored - matches structural equivalence

    Examples: `import $MODULE` (find imports); `$OBJ.$METHOD($$$)` (find method calls); `await $EXPR` (find await expressions)"""
    res = subprocess.run(
        ['ast-grep', 'run', '--pattern', pattern, '--rewrite', rewrite,
         '--lang', lang, '--update-all', '--stdin'],
        input=text, capture_output=True, text=True)
    if res.returncode: raise ValueError(res.stderr.strip())
    if not res.stdout or res.stdout == text: raise ValueError(f"No matches found for pattern: {pattern}")
    return res.stdout
```

**File:** `/usr/local/lib/python3.12/site-packages/dialoghelper/core.py`

Is that enough? You can also research additional docs.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes, this is enough. I have everything I need:

- **`ast` module**: `ast.parse`, `ast.unparse` (for reconstructing signatures), `ast.get_docstring`, `ast.get_source_segment` (for level 3 full source), and node types for `FunctionDef`, `AsyncFunctionDef`, `ClassDef`, assignments, etc.
- **fastcore's `extract_docstrings`**: already does static parsing of docstrings + param lists from source — directly useful for level 2.
- For **docments-style inline param comments** (`# comment` on param lines) statically, I'll extract them from source lines using the `lineno` attributes on `ast.arg` nodes.

A key design decision before we start: for the **signature reconstruction at level 1**, should I use `ast.unparse` (clean but loses formatting/comments) or `ast.get_source_segment` on just the `def`/`class` header lines (preserves original formatting)? 

For LLM context, `ast.unparse` is probably cleaner and more consistent — but it loses inline param comments, which we'd want to add back at level 2. `get_source_segment` on the header preserves them naturally but is harder to strip for level 1.

My suggestion: use `ast.unparse` for level 1, and for level 2 re-use the original source lines for the signature (so inline comments are preserved). Shall I proceed on that basis?

<details class='token-usage-details'><summary>$0.3571</summary>

`total=67,018 | in=65,930 | out=1,088 | cached=0.0% | cache_new=31,899 | searches=0 | $0.3571`

</details>


The original formatting is not needed (except for `docments` that probably will be good to just copy lines form source like you suggested). **Remember** this is for your benefit. You have to think what is good for you (or your LLM peers). Formatting is for humans.

Agreed. Let's start with the atomic building block: a function that reconstructs the signature string for a single function/class AST node using `ast.unparse`:

In [ ]:
import ast

def _node_sig(node):
    if isinstance(node, ast.ClassDef):
        bases = ', '.join(ast.unparse(b) for b in node.bases)
        sig = f'class {node.name}({bases}):'
    else:
        prefix = 'async def' if isinstance(node, ast.AsyncFunctionDef) else 'def'
        args, ret = ast.unparse(node.args), f' -> {ast.unparse(node.returns)}' if node.returns else ''
        sig = f'{prefix} {node.name}({args}){ret}:'
    decs = '\n'.join(f'@{ast.unparse(d)}' for d in node.decorator_list)
    return f'{decs}\n{sig}' if decs else sig

Let's verify with a quick test:

In [ ]:
_src = """
@patch
async def foo(self:MyClass, x:int, y:str='hi') -> bool: pass

class Bar(Base1, Base2): pass
"""
_tree = ast.parse(_src)
[_node_sig(n) for n in ast.walk(_tree) if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))]

["@patch\nasync def foo(self: MyClass, x: int, y: str='hi') -> bool:",
 'class Bar(Base1, Base2):']

Next atomic piece: extracting the docstring for a function/class node, and the inline param comments from source lines for level 2.

In [ ]:
def _node_doc(node): return ast.get_docstring(node) or ''

def _param_comments(node, lines):
    comments = {}
    for arg in node.args.args + node.args.posonlyargs + node.args.kwonlyargs:
        line = lines[arg.lineno - 1]
        if '#' in line: comments[arg.arg] = line.split('#', 1)[1].strip()
    return comments

In [ ]:
_src2 = """
def add(
    a:int, # first number
    b:int=0, # second number
) -> int:
    "Sum two numbers."
    return a+b
"""
_tree2 = ast.parse(_src2)
_fn = next(n for n in ast.walk(_tree2) if isinstance(n, ast.FunctionDef))
_node_doc(_fn), _param_comments(_fn, _src2.splitlines())

('Sum two numbers.', {'a': 'first number', 'b': 'second number'})

Next step: a function to format a single node at a given disclosure level, handling class bodies recursively:

In [ ]:
def _fmt_node(node, src, lines, level):
    sig = _node_sig(node)
    if level == 3: return ast.get_source_segment(src, node)
    if level == 1: return sig
    doc = _node_doc(node)
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        pcs = _param_comments(node, lines)
        pc_str = '\n'.join(f'  # {k}: {v}' for k,v in pcs.items())
        return '\n'.join(filter(None, [sig, f'  "{doc}"' if doc else '', pc_str]))
    body = '\n'.join(_fmt_node(n, src, lines, level) for n in node.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)))
    return '\n'.join(filter(None, [sig, f'  "{doc}"' if doc else '', body]))

Let's verify on a class with methods:

In [ ]:
_src3 = '''
class Foo(Base):
    "A foo class."
    def __init__(self, x:int, # the x value
                 y:str='hi'): # the y value
        "Init."
        self.x,self.y = x,y
    def bar(self) -> bool:
        "Check bar."
        return self.x > 0
'''
_tree3 = ast.parse(_src3)
_cls = next(n for n in ast.walk(_tree3) if isinstance(n, ast.ClassDef))
_lines3 = _src3.splitlines()
for lvl in (1,2,3): print(f'--- level {lvl} ---\n{_fmt_node(_cls, _src3, _lines3, lvl)}\n')

--- level 1 ---
class Foo(Base):

--- level 2 ---
class Foo(Base):
  "A foo class."
def __init__(self, x: int, y: str='hi'):
  "Init."
  # self: the x value
  # x: the x value
  # y: the y value
def bar(self) -> bool:
  "Check bar."

--- level 3 ---
class Foo(Base):
    "A foo class."
    def __init__(self, x:int, # the x value
                 y:str='hi'): # the y value
        "Init."
        self.x,self.y = x,y
    def bar(self) -> bool:
        "Check bar."
        return self.x > 0



The natural next step is the top-level `disclose` function that processes all module-level nodes:

In [ ]:
def disclose(src, level=1):
    "Return disclosure of `src` at `level` (1=sigs, 2=+docs/params, 3=full source)"
    tree, lines = ast.parse(src), src.splitlines()
    parts = []
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            parts.append(_fmt_node(node, src, lines, level))
        elif isinstance(node, (ast.Assign, ast.AnnAssign, ast.TypeAlias)):
            parts.append(ast.get_source_segment(src, node))
    return '\n\n'.join(filter(None, parts))

Examples? Edge cases? Try to use `fastcore.test` instead of plain assertions. Remember **testing is documentation**, probably the most important one. Also, be liberal outputing intermediate results so I can check along with you.

In [ ]:
_s_mods = "x = 5\ny:int = 10\ntype Alias = list[int]"
print("--- module-level ---"); print(disclose(_s_mods, 1))

--- module-level ---
x = 5

y:int = 10

type Alias = list[int]


In [ ]:
_s_fn = "def foo(a:int, b:str='hi') -> bool: return True"
test_eq(disclose(_s_fn, 1), "def foo(a: int, b: str='hi') -> bool:")
test_eq(disclose(_s_fn, 3), "def foo(a:int, b:str='hi') -> bool: return True")

I think the output should be this:
```python
def foo(a: int, b: str='hi') -> bool:...
```

It should be valid python code (or as valid as possible), simply replacing the function body with '...'. Probably you can handle equaly well, but I think valid code will cause less coginitive dissonance for you LLMs. But you know better.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Agreed — `def foo(): ...` is valid Python and much cleaner for LLM consumption. The change is minimal: append `...` to the sig at level 1. Update `_fmt_node`:

```python
def _fmt_node(node, src, lines, level):
    "Format a single AST node at disclosure `level`"
    sig = _node_sig(node)
    if level == 3: return ast.get_source_segment(src, node)
    if level == 1: return sig + '...'
    doc = _node_doc(node)
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        pcs = _param_comments(node, lines)
        pc_str = '\n'.join(f'  # {k}: {v}' for k,v in pcs.items())
        return '\n'.join(filter(None, [sig, f'  "{doc}"' if doc else '', pc_str]))
    body = '\n'.join(_fmt_node(n, src, lines, level) for n in node.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)))
    return '\n'.join(filter(None, [sig, f'  "{doc}"' if doc else '', body]))
```

And update the test:

```python
test_eq(disclose(_s_fn, 1), "def foo(a: int, b: str='hi') -> bool:...")
test_eq(disclose(_s_fn, 3), "def foo(a:int, b:str='hi') -> bool: return True")
```

<details class='token-usage-details'><summary>$0.4229</summary>

`total=70,640 | in=68,946 | out=1,694 | cached=0.0% | cache_new=66,280 | searches=0 | $0.4229`

</details>


Hmmm, why not simply replace the function body (that I suppose is a single AST node) with the '...` token. Wouldn't that be much simpler and less fragile?

Good point — `ast.unparse` on a modified deepcopy is much cleaner. We can drop `_node_sig` entirely:

In [ ]:
import copy

def _stub(node, keep_doc=False):
    "Replace function/method bodies with `...`, recursing into class bodies"
    node = copy.deepcopy(node)
    if isinstance(node, ast.ClassDef):
        node.body = [_stub(n, keep_doc) if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) else n for n in node.body]
    else:
        doc = [node.body[0]] if keep_doc and ast.get_docstring(node) else []
        node.body = doc + [ast.Expr(value=ast.Constant(value=...))]
    return node

Then `_fmt_node` level 1 becomes just `ast.unparse(_stub(node))`, and level 2 uses `_stub(node, keep_doc=True)` as the base. Let's verify first:

In [ ]:
_tree3 = ast.parse(_src3)
_cls = next(n for n in ast.walk(_tree3) if isinstance(n, ast.ClassDef))
print(ast.unparse(_stub(_cls)))
print('---')
print(ast.unparse(_stub(_cls, keep_doc=True)))

class Foo(Base):
    """A foo class."""

    def __init__(self, x: int, y: str='hi'):
        ...

    def bar(self) -> bool:
        ...
---
class Foo(Base):
    """A foo class."""

    def __init__(self, x: int, y: str='hi'):
        """Init."""
        ...

    def bar(self) -> bool:
        """Check bar."""
        ...


The stub approach works beautifully. Now update `_fmt_node` to use it:

In [ ]:
def _fmt_node(node, src, lines, level):
    "Format a single AST node at disclosure `level`"
    if level == 3: return ast.get_source_segment(src, node)
    if level == 1: return ast.unparse(_stub(node))
    doc = _node_doc(node)
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        pcs = _param_comments(node, lines)
        pc_str = '\n'.join(f'  # {k}: {v}' for k,v in pcs.items())
        return '\n'.join(filter(None, [ast.unparse(_stub(node, keep_doc=True)), pc_str]))
    body = '\n'.join(_fmt_node(n, src, lines, level) for n in node.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)))
    return '\n'.join(filter(None, [ast.unparse(_stub(node, keep_doc=True)), body]))

And update the tests:

In [ ]:
test_eq(disclose(_s_fn, 1), "def foo(a: int, b: str='hi') -> bool:\n    ...")
test_eq(disclose(_s_fn, 3), "def foo(a:int, b:str='hi') -> bool: return True")

In [ ]:
_s_async = "async def fetch(url:str) -> bytes:\n    'Fetch url.'\n    pass"
test_eq(disclose(_s_async, 1), "async def fetch(url: str) -> bytes:\n    ...")

In [ ]:
_s_patch = "@patch\ndef foo(self:MyClass, x:int): pass"
test_eq(disclose(_s_patch, 1), "@patch\ndef foo(self: MyClass, x: int):\n    ...")

In [ ]:
_s_doc = '''
def add(
    a:int, # first
    b:int=0, # second
) -> int:
    "Sum two numbers."
    return a+b
'''
test_eq(disclose(_s_doc, 1), "def add(a: int, b: int=0) -> int:\n    ...")
print("--- fn level 2 ---"); print(disclose(_s_doc, 2))
print("--- fn level 3 ---"); print(disclose(_s_doc, 3))

--- fn level 2 ---
def add(a: int, b: int=0) -> int:
    """Sum two numbers."""
    ...
  # a: first
  # b: second
--- fn level 3 ---
def add(
    a:int, # first
    b:int=0, # second
) -> int:
    "Sum two numbers."
    return a+b


Level 2 is not working right, the docments aren't it their place.

The problem is that inline param comments need to be in the signature itself, not appended after. We should drop `_param_comments` and instead use original source lines for the signature at level 2 — same idea as level 3 but replacing the body. Let's add a helper and rewrite `_fmt_node`:

In [ ]:
def _sig_src(node, src):
    "Get original source lines for just the signature (decorators thru colon)"
    ls = src.splitlines()
    start = node.decorator_list[0].lineno if node.decorator_list else node.lineno
    return '\n'.join(ls[start-1:node.body[0].lineno-1])

def _fmt_node(node, src, lines, level):
    "Format a single AST node at disclosure `level`"
    if level == 3: return ast.get_source_segment(src, node)
    if level == 1: return ast.unparse(_stub(node))
    sig, doc, ind = _sig_src(node, src), _node_doc(node), ' ' * node.body[0].col_offset
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        body = f'{ind}"""{doc}"""\n{ind}...' if doc else f'{ind}...'
        return f'{sig}\n{body}'
    hdr = f'{sig}\n{ind}"""{doc}"""' if doc else sig
    methods = '\n'.join(_fmt_node(n, src, lines, level) for n in node.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)))
    return '\n'.join(filter(None, [hdr, methods]))

In [ ]:
_s_doc = '''
def add(
    a:int, # first
    b:int=0, # second
) -> int:
    "Sum two numbers."
    return a+b
'''
test_eq(disclose(_s_doc, 1), "def add(a: int, b: int=0) -> int:\n    ...")
print("--- fn level 2 ---"); print(disclose(_s_doc, 2))
print("--- fn level 3 ---"); print(disclose(_s_doc, 3))

--- fn level 2 ---
def add(
    a:int, # first
    b:int=0, # second
) -> int:
    """Sum two numbers."""
    ...
--- fn level 3 ---
def add(
    a:int, # first
    b:int=0, # second
) -> int:
    "Sum two numbers."
    return a+b


In [ ]:
_s_cls = '''
class Foo(Base):
    "A foo."
    def __init__(self,
                 x:int, # x val
                 y:str='hi'): # y val
        "Init."
        self.x,self.y = x,y
    def bar(self) -> bool:
        "Check bar."
        return self.x > 0
    def _priv(self): pass
'''
print("--- class level 1 ---"); print(disclose(_s_cls, 1))
print("--- class level 2 ---"); print(disclose(_s_cls, 2))

--- class level 1 ---


class Foo(Base):
    """A foo."""

    def __init__(self, x: int, y: str='hi'):
        ...

    def bar(self) -> bool:
        ...

    def _priv(self):
        ...
--- class level 2 ---
class Foo(Base):
    """A foo."""
    def __init__(self,
                 x:int, # x val
                 y:str='hi'): # y val
        """Init."""
        ...
    def bar(self) -> bool:
        """Check bar."""
        ...

                     ...


The cases I'm most curious to check with you: (a) whether `self` incorrectly picks up the param comment from the next arg on the same line, (b) whether class method signatures are indented at level 2, and (c) the `type Alias` handling.

Re-check once more, but I think is working correctly, right?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Not quite — there are two issues visible in the class level 2 output:

1. **One-liner methods** (`_priv(self): pass`): `_sig_src` returns `''` because `node.body[0].lineno == node.lineno`, producing the mangled `\n                     ...` at the end.

2. **Method indentation inside class at level 2**: methods appear at column 0 instead of being indented inside the class body, which would hurt LLM readability.

The `self` param comment and `type Alias` handling both look correct though. Shall I fix these two bugs?

<details class='token-usage-details'><summary>$0.0721</summary>

`total=73,505 | in=72,347 | out=1,158 | cached=95.7% | cache_new=721 | searches=0 | $0.0721`

</details>


In [ ]:
def _sig_src(node, src):
    "Get original source lines for just the signature (decorators thru colon)"
    ls = src.splitlines()
    start = node.decorator_list[0].lineno if node.decorator_list else node.lineno
    body_line = node.body[0].lineno - 1
    if start - 1 == body_line: return ls[start-1][:node.body[0].col_offset].rstrip()
    return '\n'.join(ls[start-1:body_line])

Let's verify:

In [ ]:
print("--- class level 2 ---"); print(disclose(_s_cls, 2))

--- class level 2 ---
class Foo(Base):
    """A foo."""
    def __init__(self,
                 x:int, # x val
                 y:str='hi'): # y val
        """Init."""
        ...
    def bar(self) -> bool:
        """Check bar."""
        ...
    def _priv(self):
                     ...


Almost, the last Ellipsis is not correct, right?

Yes — for one-liners like `def _priv(self): pass`, `node.body[0].col_offset` is the column of `pass` on that line (e.g. 21), not the proper body indentation. Fix is to use `node.col_offset + 4` for one-liners:

In [ ]:
def _fmt_node(node, src, lines, level):
    "Format a single AST node at disclosure `level`"
    if level == 3: return ast.get_source_segment(src, node)
    if level == 1: return ast.unparse(_stub(node))
    sig = _sig_src(node, src)
    doc = _node_doc(node)
    one_liner = node.body[0].lineno == node.lineno
    ind = ' ' * (node.col_offset + 4 if one_liner else node.body[0].col_offset)
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        body = f'{ind}"""{doc}"""\n{ind}...' if doc else f'{ind}...'
        return f'{sig}\n{body}'
    hdr = f'{sig}\n{ind}"""{doc}"""' if doc else sig
    methods = '\n'.join(_fmt_node(n, src, lines, level) for n in node.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)))
    return '\n'.join(filter(None, [hdr, methods]))

In [ ]:
print("--- class level 2 ---"); print(disclose(_s_cls, 2))

--- class level 2 ---
class Foo(Base):
    """A foo."""
    def __init__(self,
                 x:int, # x val
                 y:str='hi'): # y val
        """Init."""
        ...
    def bar(self) -> bool:
        """Check bar."""
        ...
    def _priv(self):
        ...


Good. Some minor improvements to save context tokens. Why don't we use simple regex in the return to strip vertical spaces and to convert this pattern `:\n    ...\n` (in one liners or function w/out docstring) to `:...\n`? Probably you can think of similar harmless optimizations.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

A small `_compact` post-processor applied in `disclose`:

```python
import re

def _compact(s):
    s = re.sub(r':\n\s+\.\.\.', ':...', s)         # `:...` for empty bodies
    s = re.sub(r'"""([^"\n]+)"""', r'"\1"', s)      # triple -> single quotes for one-liners
    return re.sub(r'\n{2,}', '\n', s).strip()
```

```python
def disclose(src, level=1):
    "Return disclosure of `src` at `level` (1=sigs, 2=+docs/params, 3=full source)"
    tree, lines = ast.parse(src), src.splitlines()
    parts = []
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            parts.append(_fmt_node(node, src, lines, level))
        elif isinstance(node, (ast.Assign, ast.AnnAssign, ast.TypeAlias)):
            parts.append(ast.get_source_segment(src, node))
    return _compact('\n\n'.join(filter(None, parts)))
```

<details class='token-usage-details'><summary>$0.0660</summary>

`total=74,862 | in=73,656 | out=1,206 | cached=98.3% | cache_new=433 | searches=0 | $0.0660`

</details>


Looks good. But we have to tighten the post processors. Remember this is arbitrary Python code, we can have markdown, triple backticks, code inside docstring, etc. For example, for the Ellipsis it'd probably good to check for end of line too. We must check the postprocessor thoroughsly. Even is only for documentation and LLM usage, it should be as correct as possible.

The fundamental issue is that regex on arbitrary Python text is inherently fragile — docstrings can contain anything. The right fix is to **move compaction to construction time** in `_fmt_node`, so we never regex over content we don't fully control. `_compact` should only do the safe blank-line collapse:

In [ ]:
def _compact(s): return re.sub(r'\n{2,}', '\n', s).strip()

And handle `:...` and quote style structurally in `_fmt_node`:

In [ ]:
def _fmt_doc(doc, ind):
    "Format docstring with correct quoting"
    q = '"""' if '\n' in doc else '"'
    return f'{ind}{q}{doc}{q}'

def _fmt_node(node, src, lines, level):
    "Format a single AST node at disclosure `level`"
    if level == 3: return ast.get_source_segment(src, node)
    if level == 1: return ast.unparse(_stub(node))
    sig = _sig_src(node, src)
    doc = _node_doc(node)
    one_liner = node.body[0].lineno == node.lineno
    ind = ' ' * (node.col_offset + 4 if one_liner else node.body[0].col_offset)
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        return f'{sig}...' if not doc else f'{sig}\n{_fmt_doc(doc, ind)}\n{ind}...'
    hdr = f'{sig}\n{_fmt_doc(doc, ind)}' if doc else sig
    methods = '\n'.join(_fmt_node(n, src, lines, level) for n in node.body
                        if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)))
    return '\n'.join(filter(None, [hdr, methods]))

Now let's add tests for the tricky cases:

In [ ]:
_s_tricky = '''
def foo():
    """
    Example::

        :\\n    ...

    ```python
    x = 1
    ```
    """
    pass

def bar(): pass

def baz():
    """Has "quotes" and \\'escaped\\' stuff."""
    pass
'''
print("--- tricky level 2 ---"); print(disclose(_s_tricky, 2))
print("--- tricky level 3 ---"); print(disclose(_s_tricky, 3))

--- tricky level 2 ---
def foo():
    """Example::

    :
...

```python
x = 1
```"""
    ...

def bar():...

def baz():
    "Has "quotes" and 'escaped' stuff."
    ...
--- tricky level 3 ---
def foo():
    """
    Example::

        :\n    ...

    ```python
    x = 1
    ```
    """
    pass

def bar(): pass

def baz():
    """Has "quotes" and \'escaped\' stuff."""
    pass


Two bugs in the output: `_fmt_doc` breaks when the docstring contains `"`, and multiline docstrings lose line indentation. Fix both in `_fmt_doc`:

In [ ]:
def _fmt_doc(doc, ind):
    if '\n' in doc: return f'{ind}"""{doc.replace(chr(10), chr(10)+ind)}"""'
    return f'{ind}"""{doc}"""'

And verify the tricky cases again:

In [ ]:
print("--- tricky level 2 ---"); print(disclose(_s_tricky, 2))

--- tricky level 2 ---
def foo():
    """Example::
    
        :
    ...
    
    ```python
    x = 1
    ```"""
    ...

def bar():...

def baz():
    """Has "quotes" and 'escaped' stuff."""
    ...


Let's lock in the current behaviour with proper `test_eq` tests for the cases we've been visually checking:

In [ ]:
print("--- class level 1 ---"); print(disclose(_s_cls, 1))
print("--- class level 2 ---"); print(disclose(_s_cls, 2))

--- class level 1 ---


class Foo(Base):
    """A foo."""

    def __init__(self, x: int, y: str='hi'):
        ...

    def bar(self) -> bool:
        ...

    def _priv(self):
        ...
--- class level 2 ---
class Foo(Base):
    """A foo."""
    def __init__(self,
                 x:int, # x val
                 y:str='hi'): # y val
        """Init."""
        ...
    def bar(self) -> bool:
        """Check bar."""
        ...
    def _priv(self):...


In [ ]:
test_eq(disclose(_s_fn, 1), "def foo(a: int, b: str='hi') -> bool:...")
test_eq(disclose(_s_fn, 3), "def foo(a:int, b:str='hi') -> bool: return True")
test_eq(disclose(_s_async, 1), "async def fetch(url: str) -> bytes:...")
test_eq(disclose(_s_patch, 1), "@patch\ndef foo(self: MyClass, x: int):...")
test_eq(disclose("def bar(): pass", 2), "def bar():...")
test_eq(disclose(_s_mods, 1), "x = 5\ny:int = 10\ntype Alias = list[int]")

AssertionError: ==:
def foo(a: int, b: str='hi') -> bool:
    ...
def foo(a: int, b: str='hi') -> bool:...

The level 1 test fails because `ast.unparse(_stub(node))` always produces `:\n    ...`. Since this is output we fully control (not user source), it's safe to apply a targeted regex just there:

In [ ]:
def _fmt_node(node, src, lines, level):
    "Format a single AST node at disclosure `level`"
    if level == 3: return ast.get_source_segment(src, node)
    if level == 1: return re.sub(r':\n +\.\.\.', ':...', ast.unparse(_stub(node)))
    sig = _sig_src(node, src)
    doc = _node_doc(node)
    one_liner = node.body[0].lineno == node.lineno
    ind = ' ' * (node.col_offset + 4 if one_liner else node.body[0].col_offset)
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        return f'{sig}...' if not doc else f'{sig}\n{_fmt_doc(doc, ind)}\n{ind}...'
    hdr = f'{sig}\n{_fmt_doc(doc, ind)}' if doc else sig
    methods = '\n'.join(_fmt_node(n, src, lines, level) for n in node.body
                        if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)))
    return '\n'.join(filter(None, [hdr, methods]))

And update the class level 1 test expectation — re-run first to see what it currently produces, then lock it in:

In [ ]:
print(repr(disclose(_s_cls, 1)))

'class Foo(Base):\n    """A foo."""\n\n    def __init__(self, x: int, y: str=\'hi\'):...\n\n    def bar(self) -> bool:...\n\n    def _priv(self):...'


In [ ]:
test_eq(disclose(_s_fn, 1), "def foo(a: int, b: str='hi') -> bool:...")
test_eq(disclose(_s_fn, 3), "def foo(a:int, b:str='hi') -> bool: return True")
test_eq(disclose(_s_async, 1), "async def fetch(url: str) -> bytes:...")
test_eq(disclose(_s_patch, 1), "@patch\ndef foo(self: MyClass, x: int):...")
test_eq(disclose("def bar(): pass", 2), "def bar():...")
test_eq(disclose(_s_mods, 1), "x = 5\ny:int = 10\ntype Alias = list[int]")

AssertionError: ==:
x = 5

y:int = 10

type Alias = list[int]
x = 5
y:int = 10
type Alias = list[int]

Almost.

`disclose` isn't calling `_compact`. One-line fix:

In [ ]:
def disclose(src:str, level=1):
    "Return disclosure of `src` at `level` (1=sigs, 2=+docs/params, 3=full source)"
    tree, lines = ast.parse(src), src.splitlines()
    parts = []
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            parts.append(_fmt_node(node, src, lines, level))
        elif isinstance(node, (ast.Assign, ast.AnnAssign, ast.TypeAlias)):
            parts.append(ast.get_source_segment(src, node))
    return _compact('\n\n'.join(filter(None, parts)))

In [ ]:
test_eq(disclose(_s_fn, 1), "def foo(a: int, b: str='hi') -> bool:...")
test_eq(disclose(_s_fn, 3), "def foo(a:int, b:str='hi') -> bool: return True")
test_eq(disclose(_s_async, 1), "async def fetch(url: str) -> bytes:...")
test_eq(disclose(_s_patch, 1), "@patch\ndef foo(self: MyClass, x: int):...")
test_eq(disclose("def bar(): pass", 2), "def bar():...")
test_eq(disclose(_s_mods, 1), "x = 5\ny:int = 10\ntype Alias = list[int]")

All tests pass. Let's lock in the class and tricky-docstring cases with `test_eq`:

In [ ]:
_cls_l1 = 'class Foo(Base):\n    """A foo."""\n\n    def __init__(self, x: int, y: str=\'hi\'):...\n\n    def bar(self) -> bool:...\n\n    def _priv(self):...'
_cls_l2 = 'class Foo(Base):\n    """A foo."""\n    def __init__(self,\n                 x:int, # x val\n                 y:str=\'hi\'): # y val\n        """Init."""\n        ...\n    def bar(self) -> bool:\n        """Check bar."""\n        ...\n    def _priv(self):...'
test_eq(disclose(_s_cls, 1), _cls_l1)
test_eq(disclose(_s_cls, 2), _cls_l2)
test_eq(disclose(_s_cls, 3), _s_cls.strip())

_tricky_l2_bar = 'def bar():...'
test_eq(disclose('def bar(): pass', 2), _tricky_l2_bar)

print("all tests passed")

AssertionError: ==:
class Foo(Base):
    """A foo."""
    def __init__(self, x: int, y: str='hi'):...
    def bar(self) -> bool:...
    def _priv(self):...
class Foo(Base):
    """A foo."""

    def __init__(self, x: int, y: str='hi'):...

    def bar(self) -> bool:...

    def _priv(self):...

The expected strings in the test were wrong — `_compact` collapses the blank lines between methods. Just fix the expectations to match actual output:

In [ ]:
_cls_l1 = 'class Foo(Base):\n    """A foo."""\n    def __init__(self, x: int, y: str=\'hi\'):...\n    def bar(self) -> bool:...\n    def _priv(self):...'
test_eq(disclose(_s_cls, 1), _cls_l1)
test_eq(disclose(_s_cls, 2), _cls_l2)
test_eq(disclose(_s_cls, 3), _s_cls.strip())
test_eq(disclose('def bar(): pass', 2), 'def bar():...')
print("all tests passed")

all tests passed


Good place to consolidate all tests into one block and also add the remaining tricky-docstring cases we only printed:

In [ ]:
test_eq(disclose(_s_fn, 1), "def foo(a: int, b: str='hi') -> bool:...")
test_eq(disclose(_s_fn, 3), "def foo(a:int, b:str='hi') -> bool: return True")
test_eq(disclose(_s_async, 1), "async def fetch(url: str) -> bytes:...")
test_eq(disclose(_s_patch, 1), "@patch\ndef foo(self: MyClass, x: int):...")
test_eq(disclose("def bar(): pass", 2), "def bar():...")
test_eq(disclose(_s_mods, 1), "x = 5\ny:int = 10\ntype Alias = list[int]")
test_eq(disclose(_s_cls, 1), _cls_l1)
test_eq(disclose(_s_cls, 2), _cls_l2)
test_eq(disclose(_s_cls, 3), _s_cls.strip())
test_eq(disclose(_s_doc, 1), "def add(a: int, b: int=0) -> int:...")
test_eq(disclose(_s_doc, 3), _s_doc.strip())

_tricky_l2 = disclose(_s_tricky, 2)
test_eq('```python' in _tricky_l2, True)   # backticks in docstring survive
test_eq('pass' in disclose(_s_tricky, 3), True)  # level 3 preserves full bodies
print("all tests passed")

all tests passed


We're missing a locked `test_eq` for `_s_doc` level 2 (only printed so far). Let's add it, then move to the next building block — applying `disclose` per cell in a dialog:

In [ ]:
_doc_l2 = "def add(\n    a:int, # first\n    b:int=0, # second\n) -> int:\n    \"\"\"Sum two numbers.\"\"\"\n    ..."
test_eq(disclose(_s_doc, 2), _doc_l2)
print("all level 2 doc tests passed")

all level 2 doc tests passed


Why don't we think first more edge/not-frequent cases. For instance, lambda function at first level. Multi-line decorators are probably fine. You have seen more edge cases in your training set that I could see in several lifes.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good time to think carefully. Here are the edge cases worth addressing, roughly by priority:

1. **Lambdas in assignments** (`f = lambda x: x*2`) — currently hit as `ast.Assign` so `get_source_segment` shows the full body at all levels. Should we stub the lambda body at levels 1–2?

2. **`*args`/`**kwargs` param comments** — `_param_comments` only covers `args`, `posonlyargs`, `kwonlyargs`; it silently drops inline comments on `*args` and `**kwargs` lines.

3. **Dataclass / NamedTuple field annotations in class body** — at level 1, `_stub`+`ast.unparse` preserves them (since `_stub` only replaces function bodies); but at level 2, `_fmt_node` for classes only recurses into methods, silently dropping fields. Inconsistency between levels 1 and 2 — and arguably the fields *are* the most important part of a dataclass signature.

4. **Module-level docstring** — the first `ast.Expr(Constant(...))` in a module is completely skipped. Useful context at level 2+.

5. **Conditional definitions** (`if sys.version_info >= ...: def foo(): ...`) — the `def` is inside an `If` node, not in `tree.body` directly, so it's silently skipped at all levels.

6. **`__dunder__` noise at level 1** — for classes, level 1 currently shows all `__init__`, `__repr__`, `__eq__` etc. which is probably useful, but worth confirming that's what you want.

7. **Imports** — currently skipped entirely. Might be useful context at level 2+ for an LLM to know what's available.

Which of these do you want to tackle, and are there any you'd rather just document as known limitations?

<details class='token-usage-details'><summary>$0.1125</summary>

`total=83,632 | in=80,322 | out=3,310 | cached=99.4% | searches=0 | $0.1125`

</details>


Wow, no, very good cases:

Good time to think carefully. Here are the edge cases worth addressing, roughly by priority:

1. **Lambdas in assignments** (`f = lambda x: x*2`) — probably not worth it currently; just document for future optimizations.

2. **`*args`/`**kwargs` param comments** — document.

3. **Dataclass / NamedTuple field annotations in class body** — this is important.

4. **Module-level docstring** — very important.

5. **Conditional definitions** (`if sys.version_info >= ...: def foo(): ...`) —  important.

6. **`__dunder__` noise at level 1** — yes, leave as is for now.

7. **Imports** — Not sure about this; what do you think, is it useful for you to learn about a module?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

On imports: yes, useful at level 2+. They tell an LLM what's in scope, what style the code uses (`from fastcore.utils import *` vs explicit), and hints at architecture. At level 1 they'd just be noise since you're scanning structure. I'd include them at level 2 and 3 only.

So the priority order I'd suggest: **#4** (module docstring, trivial), **#3** (dataclass fields, moderate), **#5** (conditional defs, requires a recursive walker), then imports. Shall I start with #4?

For module docstring, the fix is a one-liner in `disclose` — detect the first node being an `ast.Expr(Constant)` and prepend it at level 2+:

```python
def disclose(src:str, level=1):
    "Return disclosure of `src` at `level` (1=sigs, 2=+docs/params, 3=full source)"
    tree, lines = ast.parse(src), src.splitlines()
    parts = []
    for i,node in enumerate(tree.body):
        if i==0 and isinstance(node, ast.Expr) and isinstance(node.value, ast.Constant) and level > 1:
            parts.append(f'"""{node.value.value}"""')
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            parts.append(_fmt_node(node, src, lines, level))
        elif isinstance(node, (ast.Assign, ast.AnnAssign, ast.TypeAlias)):
            parts.append(ast.get_source_segment(src, node))
    return _compact('\n\n'.join(filter(None, parts)))
```

<details class='token-usage-details'><summary>$0.4763</summary>

`total=81,969 | in=81,039 | out=930 | cached=0.0% | cache_new=80,656 | searches=0 | $0.4763`

</details>


On imports: yes, useful at level 2+. They tell an LLM what's in scope, what style the code uses (`from fastcore.utils import *` vs explicit), and hints at architecture. At level 1 they'd just be noise since you're scanning structure. I'd include them at level 2 and 3 only.

So the priority order I'd suggest: **#4** (module docstring, trivial), **#3** (dataclass fields, moderate), **#5** (conditional defs, requires a recursive walker), then imports. Shall I start with #4?

For module docstring, the fix is a one-liner in `disclose` — detect the first node being an `ast.Expr(Constant)` and prepend it at level 2+:

In [ ]:
def disclose(src:str, level=1):
    "Return disclosure of `src` at `level` (1=sigs, 2=+docs/params, 3=full source)"
    tree, lines = ast.parse(src), src.splitlines()
    parts = []
    for i,node in enumerate(tree.body):
        if i==0 and isinstance(node, ast.Expr) and isinstance(node.value, ast.Constant) and level > 1:
            parts.append(f'"""{node.value.value}"""')
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            parts.append(_fmt_node(node, src, lines, level))
        elif isinstance(node, (ast.Assign, ast.AnnAssign, ast.TypeAlias)):
            parts.append(ast.get_source_segment(src, node))
    return _compact('\n\n'.join(filter(None, parts)))

Let's test the module docstring, then move straight to #3 (dataclass fields):

In [ ]:
_s_moddoc = '"""This module does X."""\n\ndef foo(): pass'
test_eq(disclose(_s_moddoc, 1), "def foo():...")
test_eq(disclose(_s_moddoc, 2), '"""This module does X."""\ndef foo():...')
test_eq(disclose(_s_moddoc, 3), '"""This module does X."""\ndef foo(): pass')
print("module docstring tests passed")

module docstring tests passed


Re: dataclass. I wan't aware we were dropping assignments inside class body. That's not correct. The class namespace (that is, what's inside the class body at def tiem) should be handled like root level namespace: respect all symbols, functions or any other. Function get stripped depending of disclosure levels the other symbols (mostly assignments and defs with annotations) remain. This would also solve automatically the dataclass problem.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Exactly right — the fix is to extract a `_fmt_body` helper that mirrors what `disclose` does for `tree.body`, and use it for class bodies too:

```python
def _fmt_body(nodes, src, lines, level):
    parts = []
    for node in nodes:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            parts.append(_fmt_node(node, src, lines, level))
        elif isinstance(node, (ast.Assign, ast.AnnAssign, ast.TypeAlias)):
            parts.append(ast.get_source_segment(src, node))
    return parts

def _fmt_node(node, src, lines, level):
    "Format a single AST node at disclosure `level`"
    if level == 3: return ast.get_source_segment(src, node)
    if level == 1: return re.sub(r':\n +\.\.\.', ':...', ast.unparse(_stub(node)))
    sig = _sig_src(node, src)
    doc = _node_doc(node)
    one_liner = node.body[0].lineno == node.lineno
    ind = ' ' * (node.col_offset + 4 if one_liner else node.body[0].col_offset)
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        return f'{sig}...' if not doc else f'{sig}\n{_fmt_doc(doc, ind)}\n{ind}...'
    hdr = f'{sig}\n{_fmt_doc(doc, ind)}' if doc else sig
    return '\n'.join(filter(None, [hdr] + _fmt_body(node.body, src, lines, level)))

def disclose(src:str, level=1):
    "Return disclosure of `src` at `level` (1=sigs, 2=+docs/params, 3=full source)"
    tree, lines = ast.parse(src), src.splitlines()
    parts = []
    if tree.body and isinstance(tree.body[0], ast.Expr) and isinstance(tree.body[0].value, ast.Constant) and level > 1:
        parts.append(f'"""{tree.body[0].value.value}"""')
    parts += _fmt_body(tree.body, src, lines, level)
    return _compact('\n\n'.join(filter(None, parts)))
```

<details class='token-usage-details'><summary>$0.0769</summary>

`total=83,928 | in=82,375 | out=1,553 | cached=98.7% | cache_new=592 | searches=0 | $0.0769`

</details>


Exactly right — the fix is to extract a `_fmt_body` helper that mirrors what `disclose` does for `tree.body`, and use it for class bodies too:

In [ ]:
def _fmt_body(nodes, src, lines, level):
    parts = []
    for node in nodes:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            parts.append(_fmt_node(node, src, lines, level))
        elif isinstance(node, (ast.Assign, ast.AnnAssign, ast.TypeAlias)):
            parts.append(ast.get_source_segment(src, node))
    return parts

def _fmt_node(node, src, lines, level):
    "Format a single AST node at disclosure `level`"
    if level == 3: return ast.get_source_segment(src, node)
    if level == 1: return re.sub(r':\n +\.\.\.', ':...', ast.unparse(_stub(node)))
    sig = _sig_src(node, src)
    doc = _node_doc(node)
    one_liner = node.body[0].lineno == node.lineno
    ind = ' ' * (node.col_offset + 4 if one_liner else node.body[0].col_offset)
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        return f'{sig}...' if not doc else f'{sig}\n{_fmt_doc(doc, ind)}\n{ind}...'
    hdr = f'{sig}\n{_fmt_doc(doc, ind)}' if doc else sig
    return '\n'.join(filter(None, [hdr] + _fmt_body(node.body, src, lines, level)))

def disclose(src:str, level=1):
    "Return disclosure of `src` at `level` (1=sigs, 2=+docs/params, 3=full source)"
    tree, lines = ast.parse(src), src.splitlines()
    parts = []
    if tree.body and isinstance(tree.body[0], ast.Expr) and isinstance(tree.body[0].value, ast.Constant) and level > 1:
        parts.append(f'"""{tree.body[0].value.value}"""')
    parts += _fmt_body(tree.body, src, lines, level)
    return _compact('\n\n'.join(filter(None, parts)))

Now, once again, the fun part: testing.

In [ ]:
_s_dc = '''
from dataclasses import dataclass

@dataclass
class Point:
    "A 2D point."
    x: float
    y: float = 0.0
    def dist(self) -> float:
        "Distance from origin."
        return (self.x**2 + self.y**2)**0.5
'''
print("--- dataclass level 1 ---"); print(disclose(_s_dc, 1))
print("--- dataclass level 2 ---"); print(disclose(_s_dc, 2))

--- dataclass level 1 ---


@dataclass
class Point:
    """A 2D point."""
    x: float
    y: float = 0.0
    def dist(self) -> float:...
--- dataclass level 2 ---
@dataclass
class Point:
    """A 2D point."""
x: float
y: float = 0.0
    def dist(self) -> float:
        """Distance from origin."""
        ...


In [ ]:
_s_cls_assigns = '''
class Config:
    "App config."
    debug: bool = False
    MAX_RETRIES: int = 3
    name = "default"
    def reset(self): self.debug = False
'''
print("--- class assigns level 1 ---"); print(disclose(_s_cls_assigns, 1))
print("--- class assigns level 2 ---"); print(disclose(_s_cls_assigns, 2))

--- class assigns level 1 ---
class Config:
    """App config."""
    debug: bool = False
    MAX_RETRIES: int = 3
    name = 'default'
    def reset(self):...
--- class assigns level 2 ---
class Config:
    """App config."""
debug: bool = False
MAX_RETRIES: int = 3
name = "default"
    def reset(self):...


In [ ]:
test_eq(disclose(_s_fn, 1), "def foo(a: int, b: str='hi') -> bool:...")
test_eq(disclose(_s_fn, 3), "def foo(a:int, b:str='hi') -> bool: return True")
test_eq(disclose(_s_async, 1), "async def fetch(url: str) -> bytes:...")
test_eq(disclose(_s_patch, 1), "@patch\ndef foo(self: MyClass, x: int):...")
test_eq(disclose("def bar(): pass", 2), "def bar():...")
test_eq(disclose(_s_mods, 1), "x = 5\ny:int = 10\ntype Alias = list[int]")
test_eq(disclose(_s_cls, 1), _cls_l1)
test_eq(disclose(_s_cls, 2), _cls_l2)
test_eq(disclose(_s_cls, 3), _s_cls.strip())
test_eq(disclose(_s_moddoc, 1), "def foo():...")
test_eq(disclose(_s_moddoc, 2), '"""This module does X."""\ndef foo():...')
print("all regression tests passed")

all regression tests passed


The indentation bug — class-level assignments lose their leading spaces because `ast.get_source_segment` strips them. Fix `_fmt_body` to restore `col_offset`:

In [ ]:
def _fmt_body(nodes, src, lines, level):
    parts = []
    for node in nodes:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            parts.append(_fmt_node(node, src, lines, level))
        elif isinstance(node, (ast.Assign, ast.AnnAssign, ast.TypeAlias)):
            seg = ast.get_source_segment(src, node)
            if seg: parts.append(' ' * node.col_offset + seg)
    return parts

In [ ]:
_s_cls_assigns = '''
class Config:
    "App config."
    debug: bool = False
    MAX_RETRIES: int = 3
    name = "default"
    def reset(self): self.debug = False
'''
print("--- class assigns level 1 ---"); print(disclose(_s_cls_assigns, 1))
print("--- class assigns level 2 ---"); print(disclose(_s_cls_assigns, 2))

--- class assigns level 1 ---
class Config:
    """App config."""
    debug: bool = False
    MAX_RETRIES: int = 3
    name = 'default'
    def reset(self):...
--- class assigns level 2 ---
class Config:
    """App config."""
    debug: bool = False
    MAX_RETRIES: int = 3
    name = "default"
    def reset(self):...


Good time to re-run the full regression suite after the indentation fix, then move to #5:

In [ ]:
test_eq(disclose(_s_fn, 1), "def foo(a: int, b: str='hi') -> bool:...")
test_eq(disclose(_s_fn, 3), "def foo(a:int, b:str='hi') -> bool: return True")
test_eq(disclose(_s_async, 1), "async def fetch(url: str) -> bytes:...")
test_eq(disclose(_s_patch, 1), "@patch\ndef foo(self: MyClass, x: int):...")
test_eq(disclose("def bar(): pass", 2), "def bar():...")
test_eq(disclose(_s_mods, 1), "x = 5\ny:int = 10\ntype Alias = list[int]")
test_eq(disclose(_s_cls, 1), _cls_l1)
test_eq(disclose(_s_cls, 2), _cls_l2)
test_eq(disclose(_s_cls, 3), _s_cls.strip())
test_eq(disclose(_s_moddoc, 1), "def foo():...")
test_eq(disclose(_s_moddoc, 2), '"""This module does X."""\ndef foo():...')
test_eq(disclose(_s_doc, 2), _doc_l2)
print("all regression tests passed")

all regression tests passed


Next up is #5 — conditional defs. The approach: extend `_fmt_body` to recurse into compound statements (`If`, `Try`, `With`, `For`, `While`) to surface any defs nested inside them:

In [ ]:
def _subbodies(node):
    subs = []
    for attr in ('body', 'orelse', 'finalbody'): subs += getattr(node, attr, [])
    for h in getattr(node, 'handlers', []): subs += h.body
    return subs

_compound = (ast.If, ast.Try, ast.TryStar, ast.With, ast.For, ast.While)

def _fmt_body(nodes, src, lines, level):
    parts = []
    for node in nodes:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            parts.append(_fmt_node(node, src, lines, level))
        elif isinstance(node, (ast.Assign, ast.AnnAssign, ast.TypeAlias)):
            seg = ast.get_source_segment(src, node)
            if seg: parts.append(' ' * node.col_offset + seg)
        elif isinstance(node, _compound):
            parts += _fmt_body(_subbodies(node), src, lines, level)
    return parts

Hmmm, not sure about that. I think the chek should be only at module (root) level and class namespace. We don't want inline functions inside function body. But perhaps you have already think that.

Already handled — `_fmt_body` is only ever called from two places: `disclose` (module root) and `_fmt_node` for `ClassDef` bodies. Function bodies never go through `_fmt_body`, so inline functions inside function bodies are naturally excluded. Let's just test it:

In [ ]:
_s_cond = '''
import sys

if sys.version_info >= (3, 12):
    def new_feature(x:int) -> str: return str(x)
else:
    def new_feature(x:int) -> str: return repr(x)

def always_here(): pass
'''
print("--- cond level 1 ---"); print(disclose(_s_cond, 1))
print("--- cond level 2 ---"); print(disclose(_s_cond, 2))

--- cond level 1 ---
def new_feature(x: int) -> str:...
def new_feature(x: int) -> str:...
def always_here():...
--- cond level 2 ---
def new_feature(x:int) -> str:...
    def new_feature(x:int) -> str:...
def always_here():...


Two issues to fix: the level 2 indentation (fix `_sig_src` to strip `node.col_offset`), and duplicate names from both `if/else` branches (deduplicate by name in `_fmt_body`):

In [ ]:
def _sig_src(node, src):
    "Get original source lines for just the signature, dedented to col 0"
    ls, ind = src.splitlines(), node.col_offset
    start = node.decorator_list[0].lineno if node.decorator_list else node.lineno
    body_line = node.body[0].lineno - 1
    if start - 1 == body_line: return ls[start-1][ind:node.body[0].col_offset].rstrip()
    return '\n'.join(l[ind:] for l in ls[start-1:body_line])

In [ ]:
def _fmt_body(nodes, src, lines, level, _seen=None):
    if _seen is None: _seen = set()
    parts = []
    for node in nodes:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            if node.name not in _seen:
                _seen.add(node.name)
                parts.append(_fmt_node(node, src, lines, level))
        elif isinstance(node, (ast.Assign, ast.AnnAssign, ast.TypeAlias)):
            seg = ast.get_source_segment(src, node)
            if seg: parts.append(' ' * node.col_offset + seg)
        elif isinstance(node, _compound):
            parts += _fmt_body(_subbodies(node), src, lines, level, _seen)
    return parts

In [ ]:
_s_cond = '''
import sys

if sys.version_info >= (3, 12):
    def new_feature(x:int) -> str: return str(x)
else:
    def new_feature(x:int) -> str: return repr(x)

def always_here(): pass
'''
print("--- cond level 1 ---"); print(disclose(_s_cond, 1))
print("--- cond level 2 ---"); print(disclose(_s_cond, 2))

--- cond level 1 ---
def new_feature(x: int) -> str:...
def always_here():...
--- cond level 2 ---
def new_feature(x:int) -> str:...
def always_here():...


Right — the `if/else` is semantically load-bearing. We should preserve the conditional structure with stubbed bodies, not flatten and deduplicate. We need a `_fmt_compound` helper:

In [ ]:
def _fmt_compound(node, src, lines, level):
    ind = '    ' * (node.col_offset // 4 + 1)
    def _indent(parts): return '\n'.join(ind + l for p in parts for l in p.splitlines())
    if isinstance(node, ast.If):
        parts = [f'if {ast.unparse(node.test)}:', _indent(_fmt_body(node.body, src, lines, level))]
        if node.orelse: parts += ['else:', _indent(_fmt_body(node.orelse, src, lines, level))]
        return '\n'.join(filter(None, parts))
    return '\n'.join(_fmt_body(_subbodies(node), src, lines, level))

def _fmt_body(nodes, src, lines, level):
    parts = []
    for node in nodes:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            parts.append(_fmt_node(node, src, lines, level))
        elif isinstance(node, (ast.Assign, ast.AnnAssign, ast.TypeAlias)):
            seg = ast.get_source_segment(src, node)
            if seg: parts.append(' ' * node.col_offset + seg)
        elif isinstance(node, _compound):
            parts.append(_fmt_compound(node, src, lines, level))
    return parts

In [ ]:
_s_cond = '''
import sys

if sys.version_info >= (3, 12):
    def new_feature(x:int) -> str: return str(x)
else:
    def new_feature(x:int) -> str: return repr(x)

def always_here(): pass
'''
print("--- cond level 1 ---"); print(disclose(_s_cond, 1))
print("--- cond level 2 ---"); print(disclose(_s_cond, 2))

--- cond level 1 ---
if sys.version_info >= (3, 12):
    def new_feature(x: int) -> str:...
else:
    def new_feature(x: int) -> str:...
def always_here():...
--- cond level 2 ---
if sys.version_info >= (3, 12):
    def new_feature(x:int) -> str:...
else:
    def new_feature(x:int) -> str:...
def always_here():...


Let's run the full regression suite to make sure nothing broke after the recent refactors:

In [ ]:
test_eq(disclose(_s_fn, 1), "def foo(a: int, b: str='hi') -> bool:...")
test_eq(disclose(_s_fn, 3), "def foo(a:int, b:str='hi') -> bool: return True")
test_eq(disclose(_s_async, 1), "async def fetch(url: str) -> bytes:...")
test_eq(disclose(_s_patch, 1), "@patch\ndef foo(self: MyClass, x: int):...")
test_eq(disclose("def bar(): pass", 2), "def bar():...")
test_eq(disclose(_s_mods, 1), "x = 5\ny:int = 10\ntype Alias = list[int]")
test_eq(disclose(_s_cls, 1), _cls_l1)
test_eq(disclose(_s_cls, 2), _cls_l2)
test_eq(disclose(_s_cls, 3), _s_cls.strip())
test_eq(disclose(_s_moddoc, 1), "def foo():...")
test_eq(disclose(_s_moddoc, 2), '"""This module does X."""\ndef foo():...')
test_eq(disclose(_s_doc, 2), _doc_l2)
test_eq(disclose(_s_cls_assigns, 1), disclose(_s_cls_assigns, 1))  # smoke test
print("all regression tests passed")

AssertionError: ==:
class Foo(Base):
    """A foo."""
def __init__(self,
             x:int, # x val
             y:str='hi'): # y val
        """Init."""
        ...
def bar(self) -> bool:
        """Check bar."""
        ...
def _priv(self):...
class Foo(Base):
    """A foo."""
    def __init__(self,
                 x:int, # x val
                 y:str='hi'): # y val
        """Init."""
        ...
    def bar(self) -> bool:
        """Check bar."""
        ...
    def _priv(self):...

The indentation error again.

`_sig_src` was changed to strip `col_offset` to fix `_fmt_compound`, but that broke class body indentation. Fix: revert `_sig_src`, and use `textwrap.dedent` in `_fmt_compound` instead:

In [ ]:
import textwrap

def _sig_src(node, src):
    "Get original source lines for just the signature (decorators thru colon)"
    ls = src.splitlines()
    start = node.decorator_list[0].lineno if node.decorator_list else node.lineno
    body_line = node.body[0].lineno - 1
    if start - 1 == body_line: return ls[start-1][:node.body[0].col_offset].rstrip()
    return '\n'.join(ls[start-1:body_line])

def _fmt_compound(node, src, lines, level):
    ind = '    ' * (node.col_offset // 4 + 1)
    def _indent(parts): return '\n'.join(ind + l for p in parts for l in textwrap.dedent(p).splitlines())
    if isinstance(node, ast.If):
        parts = [f'if {ast.unparse(node.test)}:', _indent(_fmt_body(node.body, src, lines, level))]
        if node.orelse: parts += ['else:', _indent(_fmt_body(node.orelse, src, lines, level))]
        return '\n'.join(filter(None, parts))
    return '\n'.join(_fmt_body(_subbodies(node), src, lines, level))

In [ ]:
test_eq(disclose(_s_fn, 1), "def foo(a: int, b: str='hi') -> bool:...")
test_eq(disclose(_s_fn, 3), "def foo(a:int, b:str='hi') -> bool: return True")
test_eq(disclose(_s_async, 1), "async def fetch(url: str) -> bytes:...")
test_eq(disclose(_s_patch, 1), "@patch\ndef foo(self: MyClass, x: int):...")
test_eq(disclose("def bar(): pass", 2), "def bar():...")
test_eq(disclose(_s_mods, 1), "x = 5\ny:int = 10\ntype Alias = list[int]")
test_eq(disclose(_s_cls, 1), _cls_l1)
test_eq(disclose(_s_cls, 2), _cls_l2)
test_eq(disclose(_s_cls, 3), _s_cls.strip())
test_eq(disclose(_s_moddoc, 1), "def foo():...")
test_eq(disclose(_s_moddoc, 2), '"""This module does X."""\ndef foo():...')
test_eq(disclose(_s_doc, 2), _doc_l2)
test_eq(disclose(_s_cls_assigns, 1), disclose(_s_cls_assigns, 1))  # smoke test
print("all regression tests passed")

all regression tests passed


Good moment to lock in `test_eq` tests for the conditional and dataclass cases we only printed:

In [ ]:
_cond_l1 = "if sys.version_info >= (3, 12):\n    def new_feature(x: int) -> str:...\nelse:\n    def new_feature(x: int) -> str:...\ndef always_here():..."
_cond_l2 = "if sys.version_info >= (3, 12):\n    def new_feature(x:int) -> str:...\nelse:\n    def new_feature(x:int) -> str:...\ndef always_here():..."
test_eq(disclose(_s_cond, 1), _cond_l1)
test_eq(disclose(_s_cond, 2), _cond_l2)

_dc_l1 = "@dataclass\nclass Point:\n    \"\"\"A 2D point.\"\"\"\n    x: float\n    y: float = 0.0\n    def dist(self) -> float:..."
_dc_l2 = "@dataclass\nclass Point:\n    \"\"\"A 2D point.\"\"\"\n    x: float\n    y: float = 0.0\n    def dist(self) -> float:\n        \"\"\"Distance from origin.\"\"\"\n        ..."
test_eq(disclose(_s_dc, 1), _dc_l1)
test_eq(disclose(_s_dc, 2), _dc_l2)
print("conditional and dataclass tests passed")

conditional and dataclass tests passed


Next up is imports at levels 2+. The fix goes in `disclose` — collect `ast.Import` and `ast.ImportFrom` nodes and include them after the module docstring:

In [ ]:
def disclose(src:str, level=1):
    "Return disclosure of `src` at `level` (1=sigs, 2=+docs/params, 3=full source)"
    tree, lines = ast.parse(src), src.splitlines()
    parts = []
    if tree.body and isinstance(tree.body[0], ast.Expr) and isinstance(tree.body[0].value, ast.Constant) and level > 1:
        parts.append(f'"""{tree.body[0].value.value}"""')
    if level > 1:
        imps = [ast.get_source_segment(src, n) for n in tree.body if isinstance(n, (ast.Import, ast.ImportFrom))]
        if imps: parts.append('\n'.join(imps))
    parts += _fmt_body(tree.body, src, lines, level)
    return _compact('\n\n'.join(filter(None, parts)))

Let's test the imports feature and run the full regression suite:

In [ ]:
_s_imp = '''
"""A module."""
import os, sys
from pathlib import Path

def foo(): pass
'''
print("--- imports level 1 ---"); print(disclose(_s_imp, 1))
print("--- imports level 2 ---"); print(disclose(_s_imp, 2))

--- imports level 1 ---
def foo():...
--- imports level 2 ---
"""A module."""
import os, sys
from pathlib import Path
def foo():...


In [ ]:
test_eq(disclose(_s_fn, 1), "def foo(a: int, b: str='hi') -> bool:...")
test_eq(disclose(_s_fn, 3), "def foo(a:int, b:str='hi') -> bool: return True")
test_eq(disclose(_s_async, 1), "async def fetch(url: str) -> bytes:...")
test_eq(disclose(_s_patch, 1), "@patch\ndef foo(self: MyClass, x: int):...")
test_eq(disclose("def bar(): pass", 2), "def bar():...")
test_eq(disclose(_s_mods, 1), "x = 5\ny:int = 10\ntype Alias = list[int]")
test_eq(disclose(_s_cls, 1), _cls_l1)
test_eq(disclose(_s_cls, 2), _cls_l2)
test_eq(disclose(_s_cls, 3), _s_cls.strip())
test_eq(disclose(_s_moddoc, 1), "def foo():...")
test_eq(disclose(_s_moddoc, 2), '"""This module does X."""\ndef foo():...')
test_eq(disclose(_s_doc, 2), _doc_l2)
test_eq(disclose(_s_cond, 1), _cond_l1)
test_eq(disclose(_s_cond, 2), _cond_l2)
test_eq(disclose(_s_dc, 1), _dc_l1)
test_eq(disclose(_s_dc, 2), _dc_l2)
print("all regression tests passed")

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The test expectation for `_cond_l2` is stale — it now includes `import sys`. Just update it:

```python
_cond_l2 = "import sys\nif sys.version_info >= (3, 12):\n    def new_feature(x:int) -> str:...\nelse:\n    def new_feature(x:int) -> str:...\ndef always_here():..."
test_eq(disclose(_s_cond, 2), _cond_l2)
print("updated cond_l2 test passed")
```

<details class='token-usage-details'><summary>$0.0561</summary>

`total=92,304 | in=92,075 | out=229 | cached=97.6% | cache_new=534 | searches=0 | $0.0561`

</details>


The test expectation for `_cond_l2` is stale — it now includes `import sys`. Just update it:

In [ ]:
_cond_l2 = "import sys\nif sys.version_info >= (3, 12):\n    def new_feature(x:int) -> str:...\nelse:\n    def new_feature(x:int) -> str:...\ndef always_here():..."
test_eq(disclose(_s_cond, 2), _cond_l2)
print("updated cond_l2 test passed")

In [ ]:
test_eq(disclose(_s_fn, 1), "def foo(a: int, b: str='hi') -> bool:...")
test_eq(disclose(_s_fn, 3), "def foo(a:int, b:str='hi') -> bool: return True")
test_eq(disclose(_s_async, 1), "async def fetch(url: str) -> bytes:...")
test_eq(disclose(_s_patch, 1), "@patch\ndef foo(self: MyClass, x: int):...")
test_eq(disclose("def bar(): pass", 2), "def bar():...")
test_eq(disclose(_s_mods, 1), "x = 5\ny:int = 10\ntype Alias = list[int]")
test_eq(disclose(_s_cls, 1), _cls_l1)
test_eq(disclose(_s_cls, 2), _cls_l2)
test_eq(disclose(_s_cls, 3), _s_cls.strip())
test_eq(disclose(_s_moddoc, 1), "def foo():...")
test_eq(disclose(_s_moddoc, 2), '"""This module does X."""\ndef foo():...')
test_eq(disclose(_s_doc, 2), _doc_l2)
test_eq(disclose(_s_cond, 1), _cond_l1)
test_eq(disclose(_s_cond, 2), _cond_l2)
test_eq(disclose(_s_dc, 1), _dc_l1)
test_eq(disclose(_s_dc, 2), _dc_l2)
print("all regression tests passed")

Same stale expectation — `_dc_l2` now includes the import:

In [ ]:
_dc_l2 = "from dataclasses import dataclass\n@dataclass\nclass Point:\n    \"\"\"A 2D point.\"\"\"\n    x: float\n    y: float = 0.0\n    def dist(self) -> float:\n        \"\"\"Distance from origin.\"\"\"\n        ..."
test_eq(disclose(_s_dc, 2), _dc_l2)
print("updated dc_l2 test passed")

In [ ]:
test_eq(disclose(_s_fn, 1), "def foo(a: int, b: str='hi') -> bool:...")
test_eq(disclose(_s_fn, 3), "def foo(a:int, b:str='hi') -> bool: return True")
test_eq(disclose(_s_async, 1), "async def fetch(url: str) -> bytes:...")
test_eq(disclose(_s_patch, 1), "@patch\ndef foo(self: MyClass, x: int):...")
test_eq(disclose("def bar(): pass", 2), "def bar():...")
test_eq(disclose(_s_mods, 1), "x = 5\ny:int = 10\ntype Alias = list[int]")
test_eq(disclose(_s_cls, 1), _cls_l1)
test_eq(disclose(_s_cls, 2), _cls_l2)
test_eq(disclose(_s_cls, 3), _s_cls.strip())
test_eq(disclose(_s_moddoc, 1), "def foo():...")
test_eq(disclose(_s_moddoc, 2), '"""This module does X."""\ndef foo():...')
test_eq(disclose(_s_doc, 2), _doc_l2)
test_eq(disclose(_s_cond, 1), _cond_l1)
test_eq(disclose(_s_cond, 2), _cond_l2)
test_eq(disclose(_s_dc, 1), _dc_l1)
test_eq(disclose(_s_dc, 2), _dc_l2)
print("all regression tests passed")

All tests green. Let's lock in the imports test and add it to the suite, then move on to applying `disclose` at the dialog/cell level:

In [ ]:
_imp_l2 = '"""A module."""\nimport os, sys\nfrom pathlib import Path\ndef foo():...'
test_eq(disclose(_s_imp, 1), "def foo():...")
test_eq(disclose(_s_imp, 2), _imp_l2)
test_eq(disclose(_s_imp, 3), _s_imp.strip())
print("import tests passed")

`_compact` strips the blank line between imports and `def` even at level 3. Fix the expectation:

In [ ]:
_imp_l3 = '"""A module."""\nimport os, sys\nfrom pathlib import Path\ndef foo(): pass'
test_eq(disclose(_s_imp, 1), "def foo():...")
test_eq(disclose(_s_imp, 2), _imp_l2)
test_eq(disclose(_s_imp, 3), _imp_l3)
print("import tests passed")

Good — all tests green. Now let's apply `disclose` at the cell level: a function that takes a cell's source and disclosure level and returns the disclosed string, handling non-Python cells gracefully:

In [ ]:
def disclose_cell(src:str, level=1):
    "Disclose a single cell's source at `level`; returns original if not parseable Python"
    try: return disclose(src, level)
    except SyntaxError: return src

Let's test `disclose_cell` handles non-Python gracefully, then build the dialog-level function:

In [ ]:
test_eq(disclose_cell("x = 1 +", 1), "x = 1 +")   # SyntaxError -> passthrough
test_eq(disclose_cell("# just a comment", 1), "")   # valid but empty -> ok
test_eq(disclose_cell(_s_fn, 1), disclose(_s_fn, 1))
print("disclose_cell tests passed")

Now the dialog-level function that applies `disclose_cell` across all code messages:

In [ ]:
def disclose_dialog(dname:str='', level=1):
    "Return disclosed source of all code cells in dialog `dname` at `level`"
    msgs = find_msgs(dname=dname, msg_type='code', include_output=False)
    parts = [disclose_cell(m['content'], level) for m in msgs]
    return _compact('\n\n'.join(filter(None, parts)))

Add the optional argument `exported` or `only_exported`. Is SolveIt/nbdev only exported messages end up in python modules.

In [ ]:
async def disclose_dialog(dname:str='', level=1, only_exported=False):
    "Return disclosed source of all code cells in dialog `dname` at `level`"
    msgs = await find_msgs(dname=dname, msg_type='code', include_output=False, only_exp=only_exported)
    parts = [disclose_cell(m['content'], level) for m in msgs]
    return _compact('\n\n'.join(filter(None, parts)))

In [ ]:
s = await disclose_dialog(only_exported=True)
print(s)

Good output! Let's quickly check levels 2 and 3 on the current dialog to see how they look in practice:

In [ ]:
print("=== level 2 ===")
print(await disclose_dialog(only_exported=True, level=2))

The level 2 output looks great overall. One issue visible: `_mk_tag` shows a comment line in the output (`# parts = ...`) — that's a body comment leaking into `_sig_src` because it sits between the `:` and the first statement. Quick fix:

In [ ]:
def _sig_src(node, src):
    "Get original source lines for just the signature (decorators thru colon)"
    ls = src.splitlines()
    start = node.decorator_list[0].lineno if node.decorator_list else node.lineno
    body_line = node.body[0].lineno - 1
    if start - 1 == body_line: return ls[start-1][:node.body[0].col_offset].rstrip()
    sig_lines = ls[start-1:body_line]
    while sig_lines and sig_lines[-1].lstrip().startswith('#'): sig_lines.pop()
    return '\n'.join(sig_lines)

In [ ]:
print("=== level 2 ===")
print(await disclose_dialog(only_exported=True, level=2))

Run the full regression suite to confirm the `_sig_src` fix didn't break anything:

In [ ]:
test_eq(disclose(_s_fn, 1), "def foo(a: int, b: str='hi') -> bool:...")
test_eq(disclose(_s_fn, 3), "def foo(a:int, b:str='hi') -> bool: return True")
test_eq(disclose(_s_async, 1), "async def fetch(url: str) -> bytes:...")
test_eq(disclose(_s_patch, 1), "@patch\ndef foo(self: MyClass, x: int):...")
test_eq(disclose("def bar(): pass", 2), "def bar():...")
test_eq(disclose(_s_mods, 1), "x = 5\ny:int = 10\ntype Alias = list[int]")
test_eq(disclose(_s_cls, 1), _cls_l1)
test_eq(disclose(_s_cls, 2), _cls_l2)
test_eq(disclose(_s_cls, 3), _s_cls.strip())
test_eq(disclose(_s_moddoc, 1), "def foo():...")
test_eq(disclose(_s_moddoc, 2), '"""This module does X."""\ndef foo():...')
test_eq(disclose(_s_doc, 2), _doc_l2)
test_eq(disclose(_s_cond, 1), _cond_l1)
test_eq(disclose(_s_cond, 2), _cond_l2)
test_eq(disclose(_s_dc, 1), _dc_l1)
test_eq(disclose(_s_dc, 2), _dc_l2)
test_eq(disclose(_s_imp, 1), "def foo():...")
test_eq(disclose(_s_imp, 2), _imp_l2)
test_eq(disclose(_s_imp, 3), _imp_l3)
print("all regression tests passed")

The natural next step is `disclose_nb` — applying `disclose_cell` to code cells in a `.ipynb` file on disk, since the feature is intended for both dialogs and notebooks:

In [ ]:
def disclose_nb(path:str|Path, level=1):
    "Return disclosed source of all code cells in notebook at `path` at `level`"
    nb = json.loads(Path(path).read_text())
    parts = [disclose_cell(c['source'] if isinstance(c['source'], str) else ''.join(c['source']), level)
             for c in nb['cells'] if c['cell_type'] == 'code']
    return _compact('\n\n'.join(filter(None, parts)))

In [ ]:
view_dlg?

In [ ]:
s = await view_dlg()
s[:1000]

Very good. I'll test that later. Now, another optimization (but important for you and other AIs): output formatting. For Anthropic models `XMl` output is much better than JSON or plain text, right? Let's replicate `find_msg`/`view_dlg` xml-style output (that I assume is what SolveIt uses to send your context upstream) You have plenty of examples in your context surely.

Good call. The structure is clear from context — each cell becomes a tagged element with its id, wrapped in a container. We need a small helper and a `fmt` param on both functions:

In [ ]:
def _xml_cell(content, **attrs):
    attr_str = ''.join(f' {k}="{v}"' for k,v in attrs.items())
    return f'<cell{attr_str}>{content}</cell>'

async def disclose_dialog(dname:str='', level=1, only_exported=False, fmt='text'):
    "Return disclosed source of all code cells in dialog `dname` at `level`"
    msgs = await find_msgs(dname=dname, msg_type='code', include_output=False, only_exp=only_exported)
    parts = [(m['id'], disclose_cell(m['content'], level)) for m in msgs]
    parts = [(i,d) for i,d in parts if d]
    if fmt == 'xml':
        cells = '\n'.join(_xml_cell(d, id=i) for i,d in parts)
        return f'<dialog level="{level}" path="{dname or "."}">\n{cells}\n</dialog>'
    return _compact('\n\n'.join(d for _,d in parts))

def disclose_nb(path:str|Path, level=1, fmt='text'):
    "Return disclosed source of all code cells in notebook at `path` at `level`"
    nb = json.loads(Path(path).read_text())
    srcs = [c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            for c in nb['cells'] if c['cell_type'] == 'code']
    parts = [(i,d) for i,s in enumerate(srcs) if (d:=disclose_cell(s, level))]
    if fmt == 'xml':
        cells = '\n'.join(_xml_cell(d, idx=i) for i,d in parts)
        return f'<nb level="{level}" path="{path}">\n{cells}\n</nb>'
    return _compact('\n\n'.join(d for _,d in parts))

Agreed. Here's the update with `json` format support:

In [ ]:
import json

async def disclose_dialog(dname:str='', level=1, only_exported=False, fmt='text'):
    "Return disclosed source of all code cells in dialog `dname` at `level`"
    msgs = await find_msgs(dname=dname, msg_type='code', include_output=False, only_exp=only_exported)
    parts = [(m['id'], disclose_cell(m['content'], level)) for m in msgs]
    parts = [(i,d) for i,d in parts if d]
    if fmt == 'xml':
        cells = '\n'.join(_xml_cell(d, id=i) for i,d in parts)
        return f'<dialog level="{level}" path="{dname or "."}">\n{cells}\n</dialog>'
    if fmt == 'json':
        return json.dumps(dict(level=level, path=dname or ".", cells=[dict(id=i, source=d) for i,d in parts]), indent=2)
    return _compact('\n\n'.join(d for _,d in parts))

def disclose_nb(path:str|Path, level=1, fmt='text'):
    "Return disclosed source of all code cells in notebook at `path` at `level`"
    nb = json.loads(Path(path).read_text())
    srcs = [c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            for c in nb['cells'] if c['cell_type'] == 'code']
    parts = [(i,d) for i,s in enumerate(srcs) if (d:=disclose_cell(s, level))]
    if fmt == 'xml':
        cells = '\n'.join(_xml_cell(d, idx=i) for i,d in parts)
        return f'<nb level="{level}" path="{path}">\n{cells}\n</nb>'
    if fmt == 'json':
        return json.dumps(dict(level=level, path=str(path), cells=[dict(idx=i, source=d) for i,d in parts]), indent=2)
    return _compact('\n\n'.join(d for _,d in parts))

Let's test the JSON format output to verify it works correctly:

In [ ]:
s = await disclose_dialog(only_exported=True, level=1, fmt='json')
print(s[:1000])

Let me test the XML format as well to verify all three formats work:

In [ ]:
s = await disclose_dialog(only_exported=True, level=2, fmt='xml')
print(s[:1500])

Given we have a solid implementation with three output formats, shall we export the disclosure functions to the module? The key pieces to export are:

- `disclose`, `disclose_cell`, `disclose_dialog`, `disclose_nb`
- Supporting helpers: `_compact`, `_stub`, `_fmt_node`, `_fmt_body`, `_fmt_compound`, etc.
- Test strings and `test_eq` calls can stay in the notebook

We should also add the `#| export` directive to each function we want in the module. Want me to do that?

see <a class="uk-link" href="/dialog_?name=prj%2Fdutil%2Fexplorer%2Fimportdlg%2Fimportdlg" target="_blank">importdlg exploratory dialog</a>

In [ ]:
#| export
def _r(m:dict, **kwargs):
    "Add kwargs to m and filter out key/value not compatible with add_msg or update_msg"
    if kwargs: m.update(kwargs)
    if m.msg_type == 'code': m.output = ''
    return dict(content=m.content, msg_type=m.msg_type, output=m.output, is_exported=m.is_exported, skipped=m.skipped, pinned=m.pinned, i_collapsed=m.i_collapsed, o_collapsed=m.o_collapsed, heading_collapsed=m.heading_collapsed)

In [ ]:
#| export
def find_symbol_msg(msgs, sym:str) -> dict|None:
    "Find last message in msgs list containing symbol definition (module level)"
    for m in reversed(msgs):
        if m.msg_type != 'code': continue
        root = ast_py(m.content)
        for node in root.find_all(pattern='def $FUNC($$$)'):
            if node.field('name').text() == sym: return m
        for node in root.find_all(pattern='class $CLASS'):
            if node.field('name').text() == sym: return m
        for node in root.find_all(pattern='$VAR = $$$'):
            if node.field('left').text() == sym: return m
    return None

In [ ]:
#| export
async def importdlg(
    dname:str='',  # dialog to import from; relative to solveit root (if starts with /) or current dialog; empty string for current dialog
    syms:list[str]=None,  # symbol names to import (finds last def/class/assignment); if provided, regex is ignored
    ids:list[str]=None,  # message ids to import; can combine with syms; if syms or ids provided, regex is ignored
    re_include:str=None,  # regex to filter messages (content+output); only used if no syms/ids; uses re.MULTILINE|re.DOTALL
    re_exclude:str=None,  # regex to exclude from re_include results; only used if no syms/ids
    run:bool=False,  # run imported messages after insertion
    header:bool=False,  # wrap imports in collapsed header with dialog name
    heading_collapsed:bool=False  # collapse the header after creation
):
    "Import messages from dname by symbols, ids, regex filter, or all messages; inserts below current message in original order"
    msgs, to_import = await find_msgs(dname=dname, include_meta=True, include_output=True), []
    if not dname: msgs = msgs[:await msg_idx()]
    if syms or ids:
        id2idx, seen = {m.id:i for i,m in enumerate(msgs)}, set()
        if syms:
            for sym in syms:
                if (m := find_symbol_msg(msgs, sym)) and m.id not in seen: to_import.append(m); seen.add(m.id)
        if ids:
            for m in msgs:
                if m.id in ids and m.id not in seen: to_import.append(m); seen.add(m.id)
        to_import.sort(key=lambda m: id2idx[m.id])
    else:
        to_import = msgs
        if re_include: to_import = [m for m in to_import if re.search(re_include, m.content + m.output, re.MULTILINE | re.DOTALL)]
        if re_exclude: to_import = [m for m in to_import if not re.search(re_exclude, m.content + m.output, re.MULTILINE | re.DOTALL)]
    new_ids = [await add_msg(**_r(m)) for m in to_import]
    if header and new_ids: 
        hdrid = await add_msg(f"## Imported from {dname}", placement='add_before', id=new_ids[0])
        new_ids.insert(0, hdrid)
        new_ids.append(await add_msg("## ----", id=new_ids[-1]))
    # time.sleep(0.5)
    if run and new_ids: await run_msg(','.join(new_ids))
    # time.sleep(0.1)
    if heading_collapsed and header: await toggle_header(hdrid)
    return new_ids

## export -

In [ ]:
from pote.dutil import ctxusage
await ctxusage()

69670

In [ ]:
from pote.flakes import show_flakes
await show_flakes()

<div class="prose">

No warnings to report

</div>